# Graph communities for abstract questions
Inspired by microsoft paper, using leider algorithm. 


In [17]:
# 1: Connect to Neo4j and verify entity graph size

import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))

from config import NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, DATA_DIR
from neo4j import GraphDatabase
#graph imports
import json
from collections import defaultdict
 
import igraph as ig
import leidenalg
import requests
import time

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("""
        MATCH (e)
        WHERE NOT e:Chunk AND NOT e:Document
        RETURN count(e) AS entity_count
    """)
    count = result.single()["entity_count"]
    print(f"Entity nodes in graph: {count}")

Entity nodes in graph: 60884


I had some problems with duplicated entities before. Now I want to take care of at least thw abbreviations and ideally ground them in an ontology.

In [4]:
# 2: Find candidate duplicate AlgalTaxon pairs
# Pattern: "X. something" might be an abbreviation of "Xfullname something"

with driver.session() as session:
    result = session.run("""
        MATCH (a:AlgalTaxon)
        WHERE a.name CONTAINS '.'
          AND size(a.name) > 2
        RETURN a.name AS abbreviated
        ORDER BY a.name
    """)
    abbreviated = [record["abbreviated"] for record in result]

print(f"AlgalTaxon nodes with a period (potential abbreviations): {len(abbreviated)}")
print(f"\nFirst 20 examples:")
for name in abbreviated[:20]:
    print(f"  {name}")

AlgalTaxon nodes with a period (potential abbreviations): 3800

First 20 examples:
  'Bangia' 1 sp. BMW
  'Bangia' 2 sp. BGA
  . pulchella
  . robusta
  . tabulata
  A. Coffeaeformis
  A. R. Sherwood
  A. aequalis
  A. alata
  A. allophyllum collidilla
  A. allophyllum collidilla + A. allophyllum collidilla
  A. antarctica
  A. antarctica sp. nov.
  A. aponina
  A. beauvoisii
  A. boreale f. corallina
  A. brevipes var. angustata
  A. bulahemii
  A. bulnhemiy
  A. carterae


In [5]:
# Cell 3: Find all abbreviated names that have a matching full binomial

with driver.session() as session:
    result = session.run("""
        MATCH (abbr:AlgalTaxon)
        WHERE abbr.name CONTAINS '.'
        WITH abbr,
             substring(abbr.name, 0, 1) AS first_letter,
             split(abbr.name, ' ') AS parts
        WHERE size(parts) >= 2
        MATCH (full:AlgalTaxon)
        WHERE full.name STARTS WITH first_letter
          AND full.name CONTAINS ' '
          AND split(full.name, ' ')[-1] = parts[-1]
          AND NOT full.name CONTAINS '.'
          AND full.name <> abbr.name
        RETURN abbr.name AS abbreviated, 
               collect(DISTINCT full.name) AS candidates
        ORDER BY abbr.name
    """)
    matches = [(record["abbreviated"], record["candidates"]) for record in result]

# Summary stats
single_match = [m for m in matches if len(m[1]) == 1]
multi_match = [m for m in matches if len(m[1]) > 1]

print(f"Total abbreviated names with at least one match: {len(matches)}")
print(f"  Unambiguous (exactly 1 candidate): {len(single_match)}")
print(f"  Ambiguous (multiple candidates): {len(multi_match)}")

print(f"\nFirst 10 ambiguous cases:")
for abbr, candidates in multi_match[:10]:
    print(f"  {abbr}  ->  {candidates}")

Total abbreviated names with at least one match: 2932
  Unambiguous (exactly 1 candidate): 2250
  Ambiguous (multiple candidates): 682

First 10 ambiguous cases:
  A. alata  ->  ['Amphiprora alata', 'Ampiprora alata', 'Amphora alata']
  A. antarctica  ->  ['Asterochloris antarctica', 'Antithamnionella antarctica']
  A. beauvoisii  ->  ['Amphiroa beauvoisii', 'Amphirea beauvoisii']
  A. carterae  ->  ['Amphidinium carterae', 'Amphidnium carterae']
  A. convergens  ->  ['Achnanthes convergens', 'Arthrodesmus convergens']
  A. daviesii  ->  ['Audouinella daviesii', 'Auduinella daviesii', 'Acrochaetium daviesii']
  A. delicatissima  ->  ['Aphanocapsa delicatissima', 'Achnanthes delicatissima', 'Amphora delicatissima']
  A. dilatata  ->  ['Amphiroa dilatata', 'Amphiloa dilatata']
  A. ehrenbergii  ->  ['Arachnoidiscus ehrenbergii', 'Actinoptychus ehrenbergii']
  A. ephedraea  ->  ['Amphiroa ephedraea', 'Amphiron ephedraea']


In [8]:
with driver.session() as session:
    try:
        result = session.run("RETURN apoc.version() AS version")
        print(result.single()["version"])
    except Exception as e:
        print(f"APOC not available: {e}")

APOC not available: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Unknown function 'apoc.version' (line 1, column 8 (offset: 7))
"RETURN apoc.version() AS version"
        ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}


In [9]:
# Cell 4: Merge unambiguous abbreviated AlgalTaxon nodes into their full-name counterparts
# Without APOC, we handle each relationship type explicitly.
# For each pair: redirect all relationships from abbreviated -> full, then delete abbreviated.

with driver.session() as session:
    # Collect the unambiguous pairs
    result = session.run("""
        MATCH (abbr:AlgalTaxon)
        WHERE abbr.name CONTAINS '.'
        WITH abbr,
             substring(abbr.name, 0, 1) AS first_letter,
             split(abbr.name, ' ') AS parts
        WHERE size(parts) >= 2
        MATCH (full:AlgalTaxon)
        WHERE full.name STARTS WITH first_letter
          AND full.name CONTAINS ' '
          AND split(full.name, ' ')[-1] = parts[-1]
          AND NOT full.name CONTAINS '.'
          AND full.name <> abbr.name
        WITH abbr, collect(DISTINCT full) AS candidates
        WHERE size(candidates) = 1
        RETURN abbr.name AS abbr_name, candidates[0].name AS full_name
    """)
    pairs = [(record["abbr_name"], record["full_name"]) for record in result]

print(f"Found {len(pairs)} unambiguous pairs to merge")

merged = 0
errors = []

with driver.session() as session:
    for abbr_name, full_name in pairs:
        try:
            # Redirect all incoming MENTIONS edges (Chunk -> entity)
            session.run("""
                MATCH (c:Chunk)-[r:MENTIONS]->(abbr:AlgalTaxon {name: $abbr_name})
                MATCH (full:AlgalTaxon {name: $full_name})
                MERGE (c)-[:MENTIONS]->(full)
                DELETE r
            """, abbr_name=abbr_name, full_name=full_name)

            # Redirect all domain relationships where abbreviated is the SOURCE
            for rel_type in ["FOUND_IN", "PRODUCES", "STUDIED_WITH", "IDENTIFIED_BY",
                             "BELONGS_TO", "AFFECTS", "CONTAINS"]:
                session.run(f"""
                    MATCH (abbr:AlgalTaxon {{name: $abbr_name}})-[r:{rel_type}]->(target)
                    MATCH (full:AlgalTaxon {{name: $full_name}})
                    MERGE (full)-[:{rel_type} {{confidence: r.confidence}}]->(target)
                    DELETE r
                """, abbr_name=abbr_name, full_name=full_name)

            # Redirect all domain relationships where abbreviated is the TARGET
            for rel_type in ["FOUND_IN", "PRODUCES", "STUDIED_WITH", "IDENTIFIED_BY",
                             "BELONGS_TO", "AFFECTS", "CONTAINS"]:
                session.run(f"""
                    MATCH (source)-[r:{rel_type}]->(abbr:AlgalTaxon {{name: $abbr_name}})
                    MATCH (full:AlgalTaxon {{name: $full_name}})
                    MERGE (source)-[:{rel_type} {{confidence: r.confidence}}]->(full)
                    DELETE r
                """, abbr_name=abbr_name, full_name=full_name)

            # Delete the now-orphaned abbreviated node
            session.run("""
                MATCH (abbr:AlgalTaxon {name: $abbr_name})
                DELETE abbr
            """, abbr_name=abbr_name)

            merged += 1
            if merged % 100 == 0:
                print(f"  Merged {merged}/{len(pairs)}...")

        except Exception as e:
            errors.append({"abbr": abbr_name, "full": full_name, "error": str(e)})

print(f"\nDone! Merged: {merged}, Errors: {len(errors)}")
if errors:
    print("First 5 errors:")
    for err in errors[:5]:
        print(f"  {err['abbr']} -> {err['full']}: {err['error']}")

Found 2250 unambiguous pairs to merge
  Merged 100/2250...
  Merged 200/2250...
  Merged 300/2250...
  Merged 400/2250...
  Merged 500/2250...
  Merged 600/2250...
  Merged 700/2250...
  Merged 800/2250...
  Merged 900/2250...
  Merged 1000/2250...
  Merged 1100/2250...
  Merged 1200/2250...
  Merged 1300/2250...
  Merged 1400/2250...
  Merged 1500/2250...
  Merged 1600/2250...
  Merged 1700/2250...
  Merged 1800/2250...
  Merged 1900/2250...
  Merged 2000/2250...
  Merged 2100/2250...
  Merged 2200/2250...

Done! Merged: 2250, Errors: 0


In [5]:
# Cell 5: Annotate AlgalTaxon nodes with GBIF canonical taxonomy
GBIF_URL = "https://api.gbif.org/v1/species/match"

with driver.session() as session:
    # Get all AlgalTaxon names
    result = session.run("MATCH (t:AlgalTaxon) RETURN t.name AS name")
    taxa = [record["name"] for record in result]

print(f"Annotating {len(taxa)} AlgalTaxon nodes against GBIF...")

matched = 0
unmatched = 0
errors = []

with driver.session() as session:
    for i, name in enumerate(taxa):
        try:
            resp = requests.get(GBIF_URL, params={"name": name}, timeout=10)
            data = resp.json()

            # Only annotate if GBIF is reasonably confident
            if data.get("confidence", 0) >= 80 and data.get("matchType") != "NONE":
                session.run("""
                        MATCH (t:AlgalTaxon {name: $name})
                        SET t.gbif_canonical = $canonical,
                            t.gbif_status = $status,
                            t.gbif_key = $key,
                            t.gbif_confidence = $confidence,
                            t.gbif_phylum = $phylum,
                            t.gbif_class = $taxon_class,
                            t.gbif_order = $order,
                            t.gbif_family = $family,
                            t.gbif_genus = $genus
                    """, name=name,
                        canonical=data.get("canonicalName"),
                        status=data.get("status"),
                        key=data.get("usageKey"),
                        confidence=data.get("confidence"),
                        phylum=data.get("phylum"),
                        taxon_class=data.get("class"),
                        order=data.get("order"),
                        family=data.get("family"),
                        genus=data.get("genus"))
                matched += 1
            else:
                unmatched += 1

        except Exception as e:
            errors.append({"name": name, "error": str(e)})

        # GBIF asks for max 10 requests/second
        time.sleep(0.1)

        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(taxa)} - matched: {matched}, unmatched: {unmatched}")

print(f"\nDone! Matched: {matched}, Unmatched: {unmatched}, Errors: {len(errors)}")

Annotating 17769 AlgalTaxon nodes against GBIF...
  500/17769 - matched: 411, unmatched: 89
  1000/17769 - matched: 754, unmatched: 246
  1500/17769 - matched: 1026, unmatched: 474
  2000/17769 - matched: 1379, unmatched: 621
  2500/17769 - matched: 1655, unmatched: 845
  3000/17769 - matched: 1904, unmatched: 1096
  3500/17769 - matched: 2198, unmatched: 1302
  4000/17769 - matched: 2524, unmatched: 1476
  4500/17769 - matched: 2871, unmatched: 1629
  5000/17769 - matched: 3247, unmatched: 1753
  5500/17769 - matched: 3629, unmatched: 1871
  6000/17769 - matched: 4006, unmatched: 1994
  6500/17769 - matched: 4358, unmatched: 2142
  7000/17769 - matched: 4731, unmatched: 2269
  7500/17769 - matched: 5050, unmatched: 2450
  8000/17769 - matched: 5368, unmatched: 2632
  8500/17769 - matched: 5681, unmatched: 2819
  9000/17769 - matched: 6043, unmatched: 2957
  9500/17769 - matched: 6407, unmatched: 3093
  10000/17769 - matched: 6690, unmatched: 3310
  10500/17769 - matched: 7092, unmatch

In [ ]:
# 6: Export entity-to-entity edges for community detection
with driver.session() as session:
    result = session.run("""
        MATCH (e1)-[r]->(e2)
        WHERE NOT e1:Chunk AND NOT e1:Document
          AND NOT e2:Chunk AND NOT e2:Document
        RETURN e1.name AS source, e2.name AS target
    """)
    edges_raw = [(record["source"], record["target"]) for record in result]

print(f"Exported {len(edges_raw)} entity-to-entity edges")

Exported 104804 entity-to-entity edges


In [7]:
# Step 7: Build igraph graph for Leiden

# Collect all unique node names
node_set = set()
for source, target in edges_raw:
    node_set.add(source)
    node_set.add(target)

node_names = sorted(node_set)
name_to_idx = {name: idx for idx, name in enumerate(node_names)}

print(f"Unique entity nodes in domain subgraph: {len(node_names)}")

# Build graph
g = ig.Graph(directed=True)
g.add_vertices(len(node_names))
g.vs["name"] = node_names

edge_tuples = [(name_to_idx[s], name_to_idx[t]) for s, t in edges_raw]
g.add_edges(edge_tuples)

print(f"Graph: {g.vcount()} vertices, {g.ecount()} edges")
print(f"Average degree: {sum(g.degree()) / g.vcount():.1f}")

Unique entity nodes in domain subgraph: 41417
Graph: 41417 vertices, 104804 edges
Average degree: 5.1


Nodes with zero domain relationships (only connected to chunks via MENTIONS but no FOUND_IN, PRODUCES, etc.) won't appear in this export. That's fine; those isolated nodes don't contribute to community structure anyway.

I convert to undirected first because community detection cares about "which nodes cluster together," not edge direction. mode="collapse" merges parallel edges (e.g., if A->B and B->A both exist, they become one undirected edge).

In [ ]:
# Cell 8: Test of Leiden algorithm at different resolutions

g_undirected = g.as_undirected(mode="collapse")

for res in [0.5, 1.0, 1.5, 2.0]:
    partition = leidenalg.find_partition(
        g_undirected,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=res,
        seed=42
    )
    sizes = [len(c) for c in partition]
    non_singleton = sum(1 for s in sizes if s > 1)
    print(f"Resolution {res}: {len(partition)} total, {non_singleton} non-singleton, largest: {max(sizes)}")

Resolution 0.5: 946 total, 935 non-singleton, largest: 10491
Resolution 1.0: 932 total, 921 non-singleton, largest: 3857
Resolution 1.5: 950 total, 939 non-singleton, largest: 2812
Resolution 2.0: 960 total, 949 non-singleton, largest: 2170


In [10]:
# Step 9: Size distribution at two candidate resolutions

for res in [1.0, 2.0]:
    partition = leidenalg.find_partition(
        g_undirected,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=res,
        seed=42
    )
    sizes = sorted([len(c) for c in partition if len(c) > 1], reverse=True)
    
    brackets = {
        "2-5": sum(1 for s in sizes if 2 <= s <= 5),
        "6-20": sum(1 for s in sizes if 6 <= s <= 20),
        "21-100": sum(1 for s in sizes if 21 <= s <= 100),
        "101-500": sum(1 for s in sizes if 101 <= s <= 500),
        "500+": sum(1 for s in sizes if s > 500)
    }
    
    print(f"\nResolution {res}: {len(sizes)} non-singleton communities")
    for bracket, count in brackets.items():
        print(f"  {bracket}: {count}")
    print(f"  Top 5 sizes: {sizes[:5]}")


Resolution 1.0: 921 non-singleton communities
  2-5: 850
  6-20: 38
  21-100: 2
  101-500: 8
  500+: 23
  Top 5 sizes: [3857, 3783, 3644, 3429, 2495]

Resolution 2.0: 949 non-singleton communities
  2-5: 849
  6-20: 27
  21-100: 4
  101-500: 42
  500+: 27
  Top 5 sizes: [2170, 1930, 1741, 1390, 1247]


At 2.0 it's more balanced: the giants are smaller (2170 vs 3857 top), and the 101-500 bracket went from 8 to 42. Those mid-size communities are the sweet spot for producing useful summaries: big enough to represent a coherent research theme, small enough to be specific.

In [11]:
# step 10 higher resolutions

for res in [3.0, 5.0, 8.0]:
    partition = leidenalg.find_partition(
        g_undirected,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=res,
        seed=42
    )
    sizes = sorted([len(c) for c in partition if len(c) > 1], reverse=True)
    
    brackets = {
        "2-5": sum(1 for s in sizes if 2 <= s <= 5),
        "6-20": sum(1 for s in sizes if 6 <= s <= 20),
        "21-100": sum(1 for s in sizes if 21 <= s <= 100),
        "101-500": sum(1 for s in sizes if 101 <= s <= 500),
        "500+": sum(1 for s in sizes if s > 500)
    }
    
    print(f"\nResolution {res}: {len(sizes)} non-singleton communities")
    for bracket, count in brackets.items():
        print(f"  {bracket}: {count}")
    print(f"  Top 5 sizes: {sizes[:5]}")


Resolution 3.0: 983 non-singleton communities
  2-5: 848
  6-20: 26
  21-100: 11
  101-500: 77
  500+: 21
  Top 5 sizes: [1465, 1277, 1119, 1040, 879]

Resolution 5.0: 1041 non-singleton communities
  2-5: 846
  6-20: 25
  21-100: 21
  101-500: 141
  500+: 8
  Top 5 sizes: [967, 730, 721, 677, 654]

Resolution 8.0: 1130 non-singleton communities
  2-5: 848
  6-20: 25
  21-100: 78
  101-500: 178
  500+: 1
  Top 5 sizes: [739, 493, 388, 377, 368]


Resolution 8.0 is the clear winner. The 500+ bracket collapsed from 27 down to just 1, and the mid-range brackets (21-100 and 101-500) exploded: 256 communities in the 21-500 range versus 46 at resolution 2.0. Those are the communities that will produce meaningful, specific summaries. The ~848 tiny communities stay constant regardless of resolution, which implies those are genuinely tight pairs/triples that don't split further.

In [12]:
# Step 11: Final Leiden partition at resolution 8.0, inspect communities
partition = leidenalg.find_partition(
    g_undirected,
    leidenalg.RBConfigurationVertexPartition,
    resolution_parameter=8.0,
    seed=69
)

# Build community dict: comm_id -> list of entity names
communities = {}
for comm_id, members in enumerate(partition):
    if len(members) > 1:  # skip singletons
        names = [g.vs[idx]["name"] for idx in members]
        communities[comm_id] = names

print(f"Non-singleton communities: {len(communities)}")

# Show 5 largest communities with their members (first 15 names each)
for comm_id in sorted(communities, key=lambda k: len(communities[k]), reverse=True)[:5]:
    members = communities[comm_id]
    print(f"\nCommunity {comm_id} ({len(members)} entities):")
    print(f"  {members[:15]}")

Non-singleton communities: 1134

Community 0 (735 entities):
  ['15°C', 'A. aponina', 'AA', 'AY485457', 'Achnanthes sp.', 'Achnanthidium sp.', 'Actinastrum hantzschii var. fluviatile', 'Actinocyclus normanii', 'Actinocyclus octonarius', 'Actinocyclus octonarius Ehrenberg', 'Actinocyclus sp.', 'Actinocyclus sp. 0983', 'Actinoptychus senarius Ehrenberg', 'Actinoptychus sp.', 'Ago Bay']

Community 1 (530 entities):
  ['A high rhamnose-containing sulfated polysaccharide', 'A. crispata', 'A. fusiformis', 'A. ochotensis', 'Acrochaetium alariae', 'Acrosiphonia arcta', 'Acrosiphonia duriscula', 'Acrosiphonia duriuscula', 'Acrosiphonia mertensii', 'Acrosiphonia saxatilis', 'Acrosiphoniaceae', 'Acrosiphoniales', 'Acrothrix pacifica', 'Agarum gmelinii', 'Agarum pertusum']

Community 2 (433 entities):
  ['(R+C)/P', '(R+C)/P ratio', '(R+C)/P 비 계산', '(R+C)/P값', '1 m 수심', '10 m 수심', '10% 포르말린-해수 용액 고정', '1호기 취수로', '3, 6, 9, 12 m 수심', '5 m 수심', 'C. crassicaulis', 'C/P', 'C/P ratio', 'C/P 비 계산', 'C/P값'

### Community description

This cell traces each detected community back to its source documents in the knowledge graph. The goal is to identify which papers from the corpus contributed the entities that make up each community, so that their summaries can later be collected and synthesized into a community-level description.
For each non-singleton community, the cell passes the full list of entity names into a single Cypher query using UNWIND, which expands the list into individual rows within Neo4j so that all entities can be matched in one database call rather than issuing separate queries per entity. The query then traverses two relationships: first it follows MENTIONS edges backwards from entity nodes to the Chunk nodes that originally mentioned them, then it follows PART_OF edges forward from those Chunks to their parent Document nodes. The DISTINCT keyword ensures that each document filename appears only once per community, even if multiple entities in the community were mentioned across multiple chunks of the same paper.
The result is stored in a dictionary (community_papers) keyed by community ID, where each value is a list of document filenames. These filenames correspond to the JSON files in data/processed/, which contain the gemma3:1b-generated paper summaries. The mapping is many-to-many: a single community typically draws entities from multiple papers, and a single paper may contribute entities to multiple communities.
The progress counter prints every 100 communities. The sanity check at the end displays the five largest communities alongside their paper counts, which helps verify that the mapping is producing reasonable ratios — a 400-entity community should trace back to dozens or hundreds of papers, while a 2-entity community might trace to just one or two.

In [ ]:
# Step 12: Map communities to source papers via extraction cache files
CACHE_DIR = DATA_DIR / "kg_extractions"

# Build reverse lookup: entity_name -> set of document filenames
entity_to_docs = {}

category_fields = {
    "taxa": "species_name",
    "compounds": "compound_name",
    "methods": "method_name",
    "environments": "environment_name",
    "markers": "marker_name",
    "applications": "application_name"
}

for cache_file in CACHE_DIR.glob("*.json"):
    # e.g. "algae-2000-15-2-119_chunk_003.json" -> "algae-2000-15-2-119.pdf"
    doc_id = cache_file.stem.rsplit("_chunk_", 1)[0]
    filename = doc_id + ".pdf"
    
    with open(cache_file, encoding="utf-8") as f:
        extraction = json.load(f)
    
    for category, name_field in category_fields.items():
        for entity in extraction.get(category, []):
            name = entity.get(name_field)
            if name:
                if name not in entity_to_docs:
                    entity_to_docs[name] = set()
                entity_to_docs[name].add(filename)

print(f"Built entity->document index: {len(entity_to_docs)} unique entity names")

# Now map each community to its source papers
community_papers = {}
for comm_id, entity_names in communities.items():
    papers = set()
    for name in entity_names:
        if name in entity_to_docs:
            papers.update(entity_to_docs[name])
    community_papers[comm_id] = list(papers)

print(f"Mapped {len(community_papers)} communities to source papers")

for comm_id in sorted(community_papers, key=lambda k: len(communities[k]), reverse=True)[:5]:
    print(f"  Community {comm_id}: {len(communities[comm_id])} entities -> {len(community_papers[comm_id])} papers")

Built entity->document index: 62580 unique entity names
Mapped 1134 communities to source papers
  Community 0: 735 entities -> 211 papers
  Community 1: 530 entities -> 134 papers
  Community 2: 433 entities -> 76 papers
  Community 3: 401 entities -> 143 papers
  Community 4: 359 entities -> 183 papers


In [20]:
# Step13: Loads document summaries and checks coverage

PROCESSED_DIR = DATA_DIR / "processed"

doc_summaries = {}
for json_path in PROCESSED_DIR.glob("*.json"):
    with open(json_path, encoding="utf-8") as f:
        doc = json.load(f)
    filename = doc.get("filename", json_path.stem + ".pdf")
    summary = doc.get("summary", "")
    if summary:
        doc_summaries[filename] = {
            "title": doc.get("title", "Unknown"),
            "summary": summary
        }

print(f"Loaded {len(doc_summaries)} document summaries")

# Check coverage
all_papers = set()
for filenames in community_papers.values():
    all_papers.update(filenames)

covered = all_papers & set(doc_summaries.keys())
print(f"Unique papers across all communities: {len(all_papers)}")
print(f"Papers with summaries: {len(covered)}")
print(f"Coverage: {len(covered)/len(all_papers)*100:.1f}%")

Loaded 868 document summaries
Unique papers across all communities: 867
Papers with summaries: 860
Coverage: 99.2%


In [21]:
COMMUNITY_SUMMARY_PROMPT = """You are a scientific research librarian synthesizing a thematic cluster from an algae research corpus.

Below are summaries of papers whose entities were grouped together by community detection on a knowledge graph. Synthesize them into a single paragraph (3-5 sentences) that describes the shared research themes of this cluster.

Focus on:
- The species, compounds, or methods that connect these papers
- Key findings or patterns across the cluster
- The geographic or ecological scope

Be specific with species names and factual claims. Write in third person, past tense. Do not list individual papers. Respond exclusively in English. Produce only the synthesis paragraph.

Paper summaries:
{paper_summaries}"""

In [22]:
# Step 14: Generate community summaries via DeepSeek API with file-based caching

import os
import time
from openai import OpenAI

CACHE_DIR_SUMMARIES = DATA_DIR / "community_summaries"
CACHE_DIR_SUMMARIES.mkdir(parents=True, exist_ok=True)

deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
generated = 0
skipped = 0
errors = []

for comm_id, filenames in community_papers.items():
    cache_path = CACHE_DIR_SUMMARIES / f"community_{comm_id}.json"
    
    # Skip if already cached
    if cache_path.exists():
        skipped += 1
        continue
    
    # Collect available summaries for this community's papers
    available = []
    for fn in filenames:
        if fn in doc_summaries:
            entry = doc_summaries[fn]
            available.append(f"- {entry['title']}: {entry['summary']}")
    
    if not available:
        continue
    
    paper_summaries_text = "\n".join(available)
    
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": COMMUNITY_SUMMARY_PROMPT.format(
                paper_summaries=paper_summaries_text
            )}],
            temperature=0.3,
            max_tokens=512
        )
        summary = response.choices[0].message.content.strip()
        
        # Save to cache
        cache_data = {
            "community_id": comm_id,
            "summary": summary,
            "n_entities": len(communities[comm_id]),
            "n_papers": len(filenames),
            "n_papers_with_summaries": len(available),
            "entity_sample": communities[comm_id][:10],
            "resolution": 8.0
        }
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cache_data, f, indent=2, ensure_ascii=False)
        
        generated += 1
        
    except Exception as e:
        errors.appen

In [ ]:
import chromadb
# Adjust path if your ChromaDB persists elsewhere
client = chromadb.PersistentClient(path="../../data/chromadb")

# Lists all existing collections
for col in client.list_collections():
    print(f"{col.name}: {col.count()} documents")

recursive_1000_m3: 31741 documents
rsc: 25524 documents
rsc_m3: 25524 documents
recursive_100: 31741 documents
semantic_p95: 14741 documents


In [26]:
summaries_dir = Path("../../data/community_summaries")
files = sorted(summaries_dir.glob("community_*.json"))
print(f"Found {len(files)} community summary files")

if files:
    with open(files[0], "r", encoding="utf-8") as f:
        sample = json.load(f)
    print(f"\nKeys in sample: {list(sample.keys())}")
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:600])
else:
    print("No community summary files found — check the path")

Found 1128 community summary files

Keys in sample: ['community_id', 'summary', 'n_entities', 'n_papers', 'n_papers_with_summaries', 'entity_sample', 'resolution']
{
  "community_id": 0,
  "summary": "This cluster of papers is unified by a strong focus on the biodiversity, community structure, and ecological dynamics of marine and freshwater algae in Korean coastal waters, estuaries, and inland reservoirs. Key species frequently studied include the diatoms *Skeletonema costatum*, *Chaetoceros debilis*, *Stephanodiscus hantzschii*, and *Aulacoseira granulata*, the dinoflagellates *Cochlodinium polykrikoides* and *Alexandrium tamarense*, and the macroalgae *Hizikia fusiformis*, *Corallina pilulifera*, and *Ulva pertusa*. Across the cluster, researchers con


In [ ]:
import json
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions

# Load all community summaries
summaries_dir = Path("../../data/community_summaries")
summaries = []
for filepath in sorted(summaries_dir.glob("community_*.json")):
    with open(filepath, "r", encoding="utf-8") as f:
        summaries.append(json.load(f))

print(f"Loaded {len(summaries)} summaries")

# Connect to ChromaDB with the same embedding model as chunks
client = chromadb.PersistentClient(path="../../data/chromadb")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)

# Creates the collection (will error if it already exists, that's intentional)
collection = client.create_collection(
    name="community_summaries",
    embedding_function=embedding_fn
)

# Embeds and stores into a colleciton in chromadb
collection.add(
    ids=[str(s["community_id"]) for s in summaries],
    documents=[s["summary"] for s in summaries],
    metadatas=[{
        "community_id": s["community_id"],
        "n_entities": s["n_entities"],
        "n_papers": s["n_papers"],
        "resolution": s["resolution"]
    } for s in summaries]
)

print(f"Done: {collection.count()} community summaries embedded into 'community_summaries' collection")

Loaded 1128 summaries
Done — 1128 community summaries embedded into 'community_summaries' collection


ChromaDB handles embedding and similarity search internally. ChromaDB passed each summary text through bge-base-en-v1.5 and stored the resulting vectors alongside the text and metadata. Now when calling collection.query(query_texts=["..."]), it embeds the query with the same model, computes cosine distance against all 1,128 stored vectors, and returns the closest 3. No LLM, no reranking, no graph traversal. Just embed -> compare -> return.

°retrieve.py° does essentially the same thing for chunks via vectorstore.as_retriever(), but wrapped in LangChain's MultiQueryRetriever which first generates query reformulations before searching. This test cell skips all of that and hits ChromaDB directly, which is exactly how the ABSTRACT branch will work too

In [2]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="../../data/chromadb")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)

collection = client.get_collection(
    name="community_summaries",
    embedding_function=embedding_fn
)

# Test of abstract query
results = collection.query(
    query_texts=["What are the main research themes in Korean phycology?"],
    n_results=3
)

for i, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0])):
    print(f"\n--- Result {i+1} (distance: {dist:.4f}, {meta['n_entities']} entities, {meta['n_papers']} papers) ---")
    print(doc[:300])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2110.18it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Result 1 (distance: 0.2326, 92 entities, 93 papers) ---
This cluster of research is predominantly centered on the taxonomic classification, ecological dynamics, and physiological responses of phytoplankton and benthic algae in Korean aquatic environments, with a strong emphasis on diatoms and cyanobacteria. Key connecting themes include the use of morpho

--- Result 2 (distance: 0.2353, 72 entities, 47 papers) ---
This cluster of research is unified by a strong geographic focus on the Korean Peninsula and adjacent waters, with additional studies from India, Antarctica, and subtropical Asia, emphasizing the taxonomy, distribution, and ecological dynamics of diverse algal groups. Key connecting themes include t

--- Result 3 (distance: 0.2369, 401 entities, 143 papers) ---
This cluster of research predominantly focuses on the ecology, taxonomy, and distribution of phytoplankton and benthic microalgae in Korean aquatic environments, with a strong emphasis on diatoms and dinoflagellat

In [3]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="../../data/chromadb")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)
collection = client.get_collection("community_summaries", embedding_function=embedding_fn)

test_queries = [
    "What are the main research themes in Korean phycology?",
    "What compounds does Ulva pertusa produce?",
    "What is Skeletonema costatum?",
]

for q in test_queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"\n'{q}'")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"  distance: {dist:.4f} | {doc[:80]}...")


'What are the main research themes in Korean phycology?'
  distance: 0.2326 | This cluster of research is predominantly centered on the taxonomic classificati...
  distance: 0.2353 | This cluster of research is unified by a strong geographic focus on the Korean P...
  distance: 0.2369 | This cluster of research predominantly focuses on the ecology, taxonomy, and dis...

'What compounds does Ulva pertusa produce?'
  distance: 0.2780 | This cluster of papers is united by a focus on the biotechnological and bioactiv...
  distance: 0.2812 | This cluster of research is geographically centered on the Korean Peninsula and ...
  distance: 0.2823 | This cluster examines the phylogenetic relationships and genetic similarity betw...

'What is Skeletonema costatum?'
  distance: 0.3785 | This cluster of research focused on the long-term monitoring of phytoplankton co...
  distance: 0.3957 | This cluster of research focused on long-term phytoplankton community dynamics i...
  distance: 0.3973 | Thi

The threshold practically sets itself. Abstract query lands at ~0.23, relational at ~0.28, simple at ~0.38. A cutoff of 0.30 cleanly separates the first two (where community context adds value) from the simple factual query (where it'd just be noise).

The distance is cosine distance between the query embedding and each community summary embedding ;lower means more similar. What the test showed is that abstract and relational queries both land close to relevant community summaries (0.23–0.28) while simple factual queries don't (0.38+). That makes intuitive sense: "What compounds does Ulva pertusa produce?" is a relational question, but a community summary about biotechnological applications of Korean macroalgae is genuinely useful context for answering it, because it gives the generator a broader picture of which compounds have been studied and why.